# Modelo de Classificação: Predição de Popularidade Literária
Este notebook documenta o desenvolvimento de um algoritmo de Aprendizado de Máquina Supervisionado cujo objetivo é classificar o potencial de popularidade de obras literárias com base em seus metadados estruturais.

## 1. Carregamento e Inspeção Inicial dos dados
Nesta etapa, realizamos a importação dos conjuntos de dados brutos e pré-processados para avaliar sua dimensionalidade e identificar a estrutura mais adequada para a modelagem preditiva

In [9]:
import pandas as pd

# Importação dos conjuntos de dados
df_books_clean = pd.read_parquet('../data/processed/books_clean.parquet')
df_books = pd.read_parquet('../../datasets/books.parquet')
df_data = pd.read_parquet('../../datasets/data.parquet')

# Análise de dimensionalidade
print(f"Dataset processado (books_clean): {df_books_clean.shape[0]} linhas/instâncias e {df_books_clean.shape[1]} colunas/atributos")
print(f"Dataset bruto (books): {df_books.shape[0]} linhas/instâncias e {df_books.shape[1]} colunas/atributos")
print(f"Dataset bruto (data): {df_data.shape[0]} linhas/instâncias e {df_data.shape[1]} colunas/atributos")

Dataset processado (books_clean): 84054 linhas/instâncias e 14 colunas/atributos
Dataset bruto (books): 84054 linhas/instâncias e 13 colunas/atributos
Dataset bruto (data): 100000 linhas/instâncias e 13 colunas/atributos


In [10]:
print("--- Colunas de books_clean ---")
print(df_books_clean.columns.tolist())

print("\n--- Colunas de books ---")
print(df_books.columns.tolist())

print("\n--- Colunas de data ---")
print(df_data.columns.tolist())

--- Colunas de books_clean ---
['author', 'bookformat', 'desc', 'genre', 'img', 'isbn', 'isbn13', 'link', 'pages', 'rating', 'reviews', 'title', 'totalratings', 'genre_list']

--- Colunas de books ---
['author', 'bookformat', 'desc', 'genre', 'img', 'isbn', 'isbn13', 'link', 'pages', 'rating', 'reviews', 'title', 'totalratings']

--- Colunas de data ---
['author', 'bookformat', 'desc', 'genre', 'img', 'isbn', 'isbn13', 'link', 'pages', 'rating', 'reviews', 'title', 'totalratings']


In [11]:
# Mostra as primeiras linhas do arquivo 'books'
df_books.head(3)

,author,bookformat,desc,genre,img,isbn,isbn13,link,pages,rating,reviews,title,totalratings
0,Laurence M. Hauptman,Hardcover,Reveals that several hundred thousand Indians ...,"History,Military History,Civil War,American Hi...",https://i.gr-assets.com/images/S/compressed.ph...,002914180X,9.78E+12,https://goodreads.com/book/show/1001053.Betwee...,0,3.52,5,Between Two Fires: American Indians in the Civ...,33
1,"Charlotte Fiell,Emmanuelle Dirix",Paperback,Fashion Sourcebook - 1920s is the first book i...,"Couture,Fashion,Historical,Art,Nonfiction",https://i.gr-assets.com/images/S/compressed.ph...,1906863482,9.78E+12,https://goodreads.com/book/show/10010552-fashi...,576,4.51,6,Fashion Sourcebook 1920s,41
2,Andy Anderson,Paperback,The seminal history and analysis of the Hungar...,"Politics,History",https://i.gr-assets.com/images/S/compressed.ph...,948984147,9.78E+12,https://goodreads.com/book/show/1001077.Hungar...,124,4.15,2,Hungary 56,26


## 2. Engenharia de Atributos e Definição de Variável-Alvo
O modelo propõe uma abordagem de classificação binária. A variável-alvo (target) não existe nativamente no dataset e será derivada da coluna contínua `totalratings` (total de avaliações recebidas pelo livro).

In [12]:
df = df_books_clean.copy()

# 1. Convertendo colunas númericas que podem estar como texto para números reais
colunas_numericas = ['pages', 'rating', 'reviews', 'totalratings']
for col in colunas_numericas:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# 2. Vamos ver o resumo estatístico do total de avaliações (totalratings)
print('--- Estatísticas de Total de Avaliações ---')
print(df['totalratings'].describe())

# 3. Verificando o problema das páginas zeradas que encontramos
livros_sem_pagina = df[df['pages'] == 0].shape[0]
print(f"\nAlerta: Encontramos {livros_sem_pagina} livros com 0 páginas no dataset!")

--- Estatísticas de Total de Avaliações ---
count    8.405400e+04
mean     3.533681e+03
std      3.962564e+04
min      0.000000e+00
25%      5.700000e+01
50%      2.210000e+02
75%      1.000000e+03
max      3.819326e+06
Name: totalratings, dtype: float64

Alerta: Encontramos 4300 livros com 0 páginas no dataset!


### Justificativa Estatística do Ponto de Corte (Threshold)
A análise descritiva revela uma forte assimetria à direita na distribuição de avaliações. Embora a média seja de aproximadamente 3.533 avaliações, a mediana (50%) é de apenas 221. Isso reflete o comportamento de "cauda longa" do mercado editorial, onde poucos best-sellers acumulam milhões de avaliações.

Para evitar um desbalanceamento extremo de classes, adotamos o 3º quartil (Q3) como limiar lógico. Obras pertencentes aos 25% mais avaliados da plataforma (>= 1.000 avaliações) serão classificadas como **Alta Popularidade (Classe 1)**, enquanto o restante será classificado como **Nicho/Normal (Classe 0)**.

In [13]:
# 1. Tratamento de inconsistêncioas e valores ausentes (Drop NaN e páginas zeradas)
df_limpo = df[df['pages'] > 0].copy()
df_limpo = df_limpo.dropna(subset=['pages', 'rating', 'reviews', 'totalratings'])

print(f'Tamanho do dataset após higienização: {df_limpo.shape[0]} instâncias.')

#2. Criando a variável binária de popularidade (Target)
df_limpo['is_popular'] = (df_limpo['totalratings'] >= 1000).astype(int)

# 3. Separação entre variáveis preditores (x) e variável-alvo (y)
colunas_features = ['pages', 'rating', 'reviews']
x = df_limpo[colunas_features]
y = df_limpo['is_popular']

# 4. Verificação de balanceamento das classes
print('\n--- Proporção das Classes ---')
print(y.value_counts(normalize=True) * 100)

Tamanho do dataset após higienização: 79754 instâncias.

--- Proporção das Classes ---
is_popular
0    73.913534
1    26.086466
Name: proportion, dtype: float64


## 3. Divisão dos Dados e Treinamento do Modelo (K-Nearest Neighbors)
Utilizaremos o algoritmo K-Nearest Neighbors (KNN). Como o KNN baseia-se no cálculo de distância Euclidiana, é imperativo aplicar a padronização (Z-score) nas variáveis preditoras para garantir que atributos com ordens de grandeza maiores (como `reviews`) não dominem o cálculo de distância em relação a atributos menores (como `rating`).

In [14]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

# 1. Particionamento hold-out (70% treino / 30% teste)

x_treino, x_teste, y_treino, y_teste = train_test_split(x, y, test_size=0.3, random_state=42)

# 2. Padronização dos dados (Z-score normalization)
scaler = StandardScaler()
x_treino_escalado = scaler.fit_transform(x_treino)
x_teste_escalado = scaler.transform(x_teste)

# 3. Instanciando e ajuste do modelo
print('Iniciando o ajuste do modelo KNN (k=5)')
modelo_knn = KNeighborsClassifier(n_neighbors=5)
modelo_knn.fit(x_treino_escalado, y_treino)

# 4. Avaliação: aplicando o teste final (passo 5)
previsoes = modelo_knn.predict(x_teste_escalado)
acuracia = accuracy_score(y_teste, previsoes)

print(f"\nTreinamento Concluído.")
print(f'Acurácia Global do Modelo: {acuracia * 100:.2f}%')

Iniciando o ajuste do modelo KNN (k=5)

Treinamento Concluído.
Acurácia Global do Modelo: 92.01%


### Análise Crítica da Performance
O modelo atingiu uma acurácia global superior a 92%. Este é um resultado excelente para implementação em produção. No entanto, do ponto de vista de modelagem estrutural, é importante reconhecer a alta correlação natural entre a *feature* de entrada `reviews` (quantidade de resenhas escritas) e o *target* derivado de `totalratings` (quantidade de avaliações gerais).

## 4. Persistência do Modelo e Transformadores
Para integração do algoritmo preditivo com o front-end da aplicação, o modelo treinado e o padronizador de escala serão serializados e armazenados localmente.

In [15]:
import joblib
import os

base_dir = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))

if not os.path.exists(os.path.join(base_dir, "Machine Learning")):
    base_dir = os.getcwd() # Mantém a raiz atual

# Definição dos caminhos absolutos
caminho_modelo = os.path.join(base_dir, "Machine Learning", "models", "knn_popularidade.pkl")
caminho_scaler = os.path.join(base_dir, "Machine Learning", "models", "scaler_popularidade.pkl")

# Serialização (Exportação)
joblib.dump(modelo_knn, caminho_modelo)
joblib.dump(scaler, caminho_scaler)

print("Artefatos do modelo (.pkl) exportados com sucesso para o diretório de produção.")

Artefatos do modelo (.pkl) exportados com sucesso para o diretório de produção.


## 5. Simulação de Inferência em Produção
Validação do pipeline de predição simulando a entrada de dados por um usuário final através da interface do aplicativo.

In [16]:
import warnings
warnings.filterwarnings('ignore')

# 1. Simulação de payload de entrada da interface (páginas, nota, resenhas)
vetor_entrada = [[450, 4.1, 120]] #páginas, nota, resenhas

# 2. Processamento da entrada pelo transformador
entrada_escalada = scaler.transform(vetor_entrada)

# 3. Predição
previsao_modelo = modelo_knn.predict(entrada_escalada)

# 4. Resposta do sistema
if previsao_modelo[0] == 1:
    print("[LOG DA APLICAÇÃO] Saída: Alta Popularidade Prevista (Classe 1).")
else: 
    print("[LOG DA APLICAÇÃO] Saída: Obra de Nicho ou Popularidade Padrão (Classe 0).")

[LOG DA APLICAÇÃO] Saída: Obra de Nicho ou Popularidade Padrão (Classe 0).
